In [2]:
import pandas as pd
df = pd.read_csv('../../02_Data/processed/real_final_ml.csv')

c:\Users\seon\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\seon\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
df = df.drop(columns=['enableBoardGameProperties','projectID','campaignGoal_usd_6m','fundsGathered_usd_6m','price_usd_6m','is_backer_0','is_backer_1',"fundedInSeconds"])

### 후진 제거법

In [4]:
import numpy as np
import pandas as pd
import warnings
import lightgbm as lgb
import re
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score



# 1. 1.5 IQR 아웃라이어 제거
target_col = 'fundsGathered_usd_1m'
q1 = df[target_col].quantile(0.25)
q3 = df[target_col].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df_filtered = df[(df[target_col] >= lower_bound) & (df[target_col] <= upper_bound)].copy()
df_filtered = df_filtered[df_filtered[target_col] > 0].reset_index(drop=True)

# 2. 로그 변환
X_full = df_filtered.drop(columns=[target_col])
y_original = df_filtered[target_col]
y_log = np.log1p(y_original)


X_full.columns = [re.sub(r'[ ,\{\}\:\"\]\[\-]', '_', col) for col in X_full.columns]

# 3. 5-Fold 
kf = KFold(n_splits=5, shuffle=True, random_state=42)
feature_importances = np.zeros(X_full.shape[1])

# 전체 변수
for train_idx, test_idx in kf.split(X_full):
    X_tr, X_te = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_tr_log = y_log.iloc[train_idx]
    
    model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X_tr, y_tr_log)
    feature_importances += model.booster_.feature_importance(importance_type='gain') / 5

df_imp = pd.DataFrame({
    'Feature': X_full.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

# 기여도 0인 변수 제거
current_features = df_imp[df_imp['Importance'] > 0.0]['Feature'].tolist()

best_mae = float('inf')
best_feature_set = []
step = 1

# 4. 후진 제거법 
while len(current_features) > 2:
    cv_te_mae = []
    X_step = X_full[current_features]
    
    for train_idx, test_idx in kf.split(X_step):
        X_tr, X_te = X_step.iloc[train_idx], X_step.iloc[test_idx]
        y_tr_log = y_log.iloc[train_idx]
        
        model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
        model.fit(X_tr, y_tr_log)
        
        te_preds = np.expm1(model.predict(X_te))
        cv_te_mae.append(mean_absolute_error(y_original.iloc[test_idx], te_preds))
        
    current_mae = np.mean(cv_te_mae)
    
    # 남은 변수 기여도 확인
    full_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    full_model.fit(X_step, y_log)
    
    step_imp = pd.DataFrame({
        'Feature': current_features,
        'Importance': full_model.booster_.feature_importance(importance_type='gain')
    }).sort_values(by='Importance', ascending=True).reset_index(drop=True)
    
    # 성능 갱신  업데이트
    if current_mae < best_mae:
        best_mae = current_mae
        best_feature_set = list(current_features)
        
    if step % 10 == 1 or len(current_features) <= 15:
        print(f"[Step {step:02d}] 남은 변수: {len(current_features)}개 -> Test MAE: {current_mae:,.2f}")
        
    # 기여도 최하위 변수 1개 드랍
    lowest_feature = step_imp.iloc[0]['Feature']
    current_features.remove(lowest_feature)
    step += 1

print(f"\n최종 최저 오차(MAE): {best_mae:,.2f}")
print(f"최종 선택 변수 개수: {len(best_feature_set)}개")

# 결과 저장
#pd.Series(best_feature_set).to_csv('backward_selected_features.csv', index=False)

c:\Users\seon\anaconda3\lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


[Step 01] 남은 변수: 109개 -> Test MAE: 29,334.98
[Step 11] 남은 변수: 99개 -> Test MAE: 29,428.82
[Step 21] 남은 변수: 89개 -> Test MAE: 28,732.03
[Step 31] 남은 변수: 79개 -> Test MAE: 28,729.36
[Step 41] 남은 변수: 69개 -> Test MAE: 29,641.68
[Step 51] 남은 변수: 59개 -> Test MAE: 29,599.91
[Step 61] 남은 변수: 49개 -> Test MAE: 29,973.39
[Step 71] 남은 변수: 39개 -> Test MAE: 30,669.79
[Step 81] 남은 변수: 29개 -> Test MAE: 30,966.13
[Step 91] 남은 변수: 19개 -> Test MAE: 31,140.86
[Step 95] 남은 변수: 15개 -> Test MAE: 30,013.24
[Step 96] 남은 변수: 14개 -> Test MAE: 29,465.75
[Step 97] 남은 변수: 13개 -> Test MAE: 30,026.22
[Step 98] 남은 변수: 12개 -> Test MAE: 29,839.88
[Step 99] 남은 변수: 11개 -> Test MAE: 29,955.78
[Step 100] 남은 변수: 10개 -> Test MAE: 29,510.71
[Step 101] 남은 변수: 9개 -> Test MAE: 29,651.40
[Step 102] 남은 변수: 8개 -> Test MAE: 29,111.51
[Step 103] 남은 변수: 7개 -> Test MAE: 28,478.25
[Step 104] 남은 변수: 6개 -> Test MAE: 28,222.86
[Step 105] 남은 변수: 5개 -> Test MAE: 30,032.20
[Step 106] 남은 변수: 4개 -> Test MAE: 30,550.71
[Step 107] 남은 변수: 3개 -> Test M

In [5]:
best_feature_set

['is_pledge_master_1',
 '중립_1',
 'Product_Question_count_1',
 'softclose',
 'likes_1',
 'campaignGoal_usd_1m']

### 옵투나 최적화

In [7]:
"""
===============================================================================
[LightGBM + Optuna 하이퍼파라미터 최적화 및 최종 평가 지표 출력]
===============================================================================
"""

import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score, median_absolute_error, mean_squared_error
import lightgbm as lgb
import numpy as np
import pandas as pd

# 최적화 변수
X_best = X_full[best_feature_set]

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    cv_mae = []
    
    for train_idx, test_idx in kf.split(X_best):
        X_tr, X_te = X_best.iloc[train_idx], X_best.iloc[test_idx]
        y_tr_log = y_log.iloc[train_idx]
        y_te_original = y_original.iloc[test_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr_log)

        preds_log = model.predict(X_te)
        preds_original = np.expm1(preds_log)

        cv_mae.append(mean_absolute_error(y_te_original, preds_original))

    return np.mean(cv_mae)

# Optuna 로그 출력 최소화
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Optuna 최적화 실행
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("최적 하이퍼파라미터 조합:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print(f"\n최저 검증 MAE: {study.best_value:,.2f}\n")


# ---------------------------------------------------------
# 최적 파라미터 기반 K-Fold 교차 검증 및 지표 출력
# ---------------------------------------------------------
results_folds = []

for train_idx, test_idx in kf.split(X_best):
    X_tr, X_te = X_best.iloc[train_idx], X_best.iloc[test_idx]
    y_tr_log = y_log.iloc[train_idx]
    
    y_tr_real = y_original.iloc[train_idx]
    y_te_real = y_original.iloc[test_idx]
    
    final_model = lgb.LGBMRegressor(**study.best_params, random_state=42, n_jobs=-1, verbose=-1)
    final_model.fit(X_tr, y_tr_log)
    
    tr_preds_real = np.expm1(final_model.predict(X_tr))
    te_preds_real = np.expm1(final_model.predict(X_te))
    
    fold_res = {
        'tr_mae': mean_absolute_error(y_tr_real, tr_preds_real),
        'te_mae': mean_absolute_error(y_te_real, te_preds_real),
        'tr_r2': r2_score(y_tr_real, tr_preds_real),
        'te_r2': r2_score(y_te_real, te_preds_real),
        'te_medae': median_absolute_error(y_te_real, te_preds_real),
        'te_rmse': np.sqrt(mean_squared_error(y_te_real, te_preds_real))
    }
    results_folds.append(fold_res)

df_folds = pd.DataFrame(results_folds)
metrics_summary = df_folds.mean()

print("최종 모델 K-Fold 교차 검증 평균 성능 지표:")
print("-" * 50)
print(f"Train MAE  : {metrics_summary['tr_mae']:,.2f}")
print(f"Test MAE   : {metrics_summary['te_mae']:,.2f}")
print(f"MAE Gap    : {metrics_summary['te_mae'] - metrics_summary['tr_mae']:,.2f}")
print("-" * 50)
print(f"Train R2   : {metrics_summary['tr_r2']:.4f}")
print(f"Test R2    : {metrics_summary['te_r2']:.4f}")
print("-" * 50)
print(f"Test MedAE : {metrics_summary['te_medae']:,.2f}")
print(f"Test RMSE  : {metrics_summary['te_rmse']:,.2f}")
print("-" * 50)

# 최종 데이터 전체 학습 모델 
final_best_model = lgb.LGBMRegressor(**study.best_params, random_state=42, n_jobs=-1, verbose=-1)
final_best_model.fit(X_best, y_log)

최적 하이퍼파라미터 조합:
  n_estimators: 300
  learning_rate: 0.025812499344723513
  max_depth: 3
  num_leaves: 23
  subsample: 0.8830023606726238
  colsample_bytree: 0.6930345892864527
  reg_alpha: 0.00793978611357863
  reg_lambda: 0.05722780242382558

최저 검증 MAE: 26,324.03

최종 모델 K-Fold 교차 검증 평균 성능 지표:
--------------------------------------------------
Train MAE  : 22,103.48
Test MAE   : 26,324.03
MAE Gap    : 4,220.54
--------------------------------------------------
Train R2   : 0.8876
Test R2    : 0.8438
--------------------------------------------------
Test MedAE : 6,806.40
Test RMSE  : 51,266.25
--------------------------------------------------


LGBMRegressor(colsample_bytree=0.6930345892864527,
              learning_rate=0.025812499344723513, max_depth=3, n_estimators=300,
              n_jobs=-1, num_leaves=23, random_state=42,
              reg_alpha=0.00793978611357863, reg_lambda=0.05722780242382558,
              subsample=0.8830023606726238, verbose=-1)